# HS Universe — Full In-Degree Dataset

Builds a complete universe of all US high schools (highest_grade_offered = 12 or 13),
then merges with `hs_in_degree.csv` so that every school has an in-degree value:
schools visited by at least one college keep their count; all other schools get 0.

**Sources:**
- Public schools: Urban Institute Education Data API — CCD Directory + CCD Enrollment by Race
- Private schools: PSS 2019-20 (for 2019) and PSS 2021-22 (for 2023)

In [1]:
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'

YEARS = [2019, 2023]

session = requests.Session()
retries = Retry(
    total=8, connect=8, read=8, status=8,
    backoff_factor=0.75,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(['GET']),
    respect_retry_after_header=True,
)
session.mount('https://', HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10))
session.headers.update({'User-Agent': 'c2i-hs-universe/1.0'})

def fetch_json(url: str) -> dict:
    for attempt in range(6):
        try:
            resp = session.get(url, timeout=(15, 120))
            resp.raise_for_status()
            return resp.json()
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError):
            if attempt == 5:
                raise
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError('Unreachable')

print('Setup complete.')

Setup complete.


## 1. Pull Public HS from CCD Directory API

Filter: `highest_grade_offered` in [12, 13].
Fields pulled match those used throughout the project pipeline.

In [2]:
# school_level string mapping (from standardize_hs.ipynb)
pub_level_map = {
    0: 'pre_k', 1: 'elementary', 2: 'middle', 3: 'high',
    4: 'other', 5: 'ungraded', 6: 'adult_ed', 7: 'secondary',
    -1: None, -2: None, -3: None,
}

ccd_dir_frames = []

for year in YEARS:
    url = f'https://educationdata.urban.org/api/v1/schools/ccd/directory/{year}/'
    rows = []
    page = 0
    while url:
        data = fetch_json(url)
        for row in data.get('results', []):
            hg = pd.to_numeric(row.get('highest_grade_offered'), errors='coerce')
            if pd.notna(hg) and int(hg) in (12, 13):
                rows.append(row)
        url = data.get('next')
        page += 1
        if page % 20 == 0:
            time.sleep(0.05)

    df_year = pd.DataFrame(rows)
    df_year['year'] = year
    ccd_dir_frames.append(df_year)
    print(f'{year}: {len(rows):,} public HS with highest_grade_offered in [12,13]')

ccd_dir = pd.concat(ccd_dir_frames, ignore_index=True)
ccd_dir.columns = [c.lower() for c in ccd_dir.columns]
print(f'\nTotal CCD rows: {len(ccd_dir):,}')
print('Available CCD columns:', [c for c in ccd_dir.columns if c in [
    'ncessch','school_name','city_location','state_location','zip_location',
    'county_code','county_name','latitude','longitude','school_level',
    'highest_grade_offered','charter','magnet','title_i_status',
    'urban_centric_locale','enrollment','teachers_fte','free_or_reduced_price_lunch'
]])

2019: 27,139 public HS with highest_grade_offered in [12,13]
2023: 28,072 public HS with highest_grade_offered in [12,13]

Total CCD rows: 55,211
Available CCD columns: ['ncessch', 'school_name', 'city_location', 'state_location', 'zip_location', 'latitude', 'longitude', 'urban_centric_locale', 'county_code', 'school_level', 'highest_grade_offered', 'title_i_status', 'charter', 'magnet', 'teachers_fte', 'free_or_reduced_price_lunch', 'enrollment']


In [3]:
# Extract and rename to hs_ columns
# Note: CCD directory API does not provide county_name — only county_code (FIPS).
# hs_ctyname will be null for public schools; it is available for private schools via PSS (PCNTNM).
field_map = {
    'ncessch':                    'hs_id',
    'school_name':                'hs_name',
    'city_location':              'hs_city',
    'state_location':             'hs_state',
    'zip_location':               'hs_zip',
    'county_code':                'hs_cty_fips',
    'latitude':                   'hs_lat',
    'longitude':                  'hs_long',
    'school_level':               '_school_level_raw',
    'highest_grade_offered':      'hs_highest_grade_offered',
    'charter':                    'hs_charter',
    'magnet':                     'hs_magnet',
    'title_i_status':             'hs_title_i_status',
    'urban_centric_locale':       'hs_urban_centric_locale',
    'enrollment':                 'hs_enrollment',
    'teachers_fte':               '_teachers_fte',
    'free_or_reduced_price_lunch':'_frl_count',
}

keep = {k: v for k, v in field_map.items() if k in ccd_dir.columns}
pub_hs = ccd_dir[['year'] + list(keep.keys())].copy().rename(columns=keep)

# Numeric coercions
for col in ['hs_highest_grade_offered','hs_charter','hs_magnet','hs_title_i_status',
            'hs_urban_centric_locale','hs_enrollment','hs_zip','hs_cty_fips',
            '_school_level_raw','_teachers_fte','_frl_count']:
    if col in pub_hs.columns:
        pub_hs[col] = pd.to_numeric(pub_hs[col], errors='coerce')

# Map school_level code → string (same as standardize_hs.ipynb)
if '_school_level_raw' in pub_hs.columns:
    pub_hs['hs_school_level'] = pub_hs['_school_level_raw'].map(
        lambda v: pub_level_map.get(int(v)) if pd.notna(v) else None
    )
    pub_hs.drop(columns=['_school_level_raw'], inplace=True)

# Derived columns
pub_hs['hs_pct_free_or_reduced_price_lunch'] = np.where(
    pub_hs['hs_enrollment'].notna() & (pub_hs['hs_enrollment'] > 0) & pub_hs['_frl_count'].notna(),
    pub_hs['_frl_count'] / pub_hs['hs_enrollment'],
    np.nan
)
pub_hs['hs_students_per_teacher'] = np.where(
    pub_hs['_teachers_fte'].notna() & (pub_hs['_teachers_fte'] > 0) & pub_hs['hs_enrollment'].notna(),
    pub_hs['hs_enrollment'] / pub_hs['_teachers_fte'],
    np.nan
)
pub_hs.drop(columns=['_frl_count', '_teachers_fte'], inplace=True, errors='ignore')

pub_hs['hs_id'] = pub_hs['hs_id'].astype('string').str.strip()
pub_hs['hs_ctyname'] = np.nan  # not available from CCD API
pub_hs['school_type'] = 'public'

# Deduplicate by (year, hs_id)
pub_hs = pub_hs.drop_duplicates(subset=['year', 'hs_id'])

print('pub_hs shape:', pub_hs.shape)
print('Columns:', pub_hs.columns.tolist())

pub_hs shape: (55211, 20)
Columns: ['year', 'hs_id', 'hs_name', 'hs_city', 'hs_state', 'hs_zip', 'hs_cty_fips', 'hs_lat', 'hs_long', 'hs_highest_grade_offered', 'hs_charter', 'hs_magnet', 'hs_title_i_status', 'hs_urban_centric_locale', 'hs_enrollment', 'hs_school_level', 'hs_pct_free_or_reduced_price_lunch', 'hs_students_per_teacher', 'hs_ctyname', 'school_type']


## 1b. Backfill magnet & title_i_status for 2023 from 2021 CCD\n\nThe CCD directory API has no `magnet` or `title_i_status` data for 2022–2023.\n2021 is the most recent year with full coverage; school designations rarely change year to year.

In [4]:
# Pull 2021 CCD directory — magnet and title_i_status only
url_2021 = 'https://educationdata.urban.org/api/v1/schools/ccd/directory/2021/'
rows_2021 = []
while url_2021:
    data = fetch_json(url_2021)
    for row in data.get('results', []):
        ncessch = str(row.get('ncessch', '')).strip()
        if ncessch:
            rows_2021.append({
                'hs_id':                 ncessch,
                'hs_magnet_2021':        row.get('magnet'),
                'hs_title_i_status_2021':row.get('title_i_status'),
            })
    url_2021 = data.get('next')
    time.sleep(0.03)

ccd_2021 = pd.DataFrame(rows_2021)
for col in ['hs_magnet_2021', 'hs_title_i_status_2021']:
    ccd_2021[col] = pd.to_numeric(ccd_2021[col], errors='coerce')

print(f'2021 CCD pulled: {len(ccd_2021):,} schools')
print(f'  magnet non-null:         {ccd_2021["hs_magnet_2021"].notna().sum():,}')
print(f'  title_i_status non-null: {ccd_2021["hs_title_i_status_2021"].notna().sum():,}')

# Backfill into pub_hs for 2023 rows only
pub_hs = pub_hs.merge(ccd_2021, on='hs_id', how='left')

mask_2023 = pub_hs['year'] == 2023
pub_hs.loc[mask_2023, 'hs_magnet'] = (
    pub_hs.loc[mask_2023, 'hs_magnet']
    .fillna(pub_hs.loc[mask_2023, 'hs_magnet_2021'])
)
pub_hs.loc[mask_2023, 'hs_title_i_status'] = (
    pub_hs.loc[mask_2023, 'hs_title_i_status']
    .fillna(pub_hs.loc[mask_2023, 'hs_title_i_status_2021'])
)
pub_hs.drop(columns=['hs_magnet_2021', 'hs_title_i_status_2021'], inplace=True)

still_null_mag = pub_hs.loc[mask_2023, 'hs_magnet'].isna().sum()
still_null_ti  = pub_hs.loc[mask_2023, 'hs_title_i_status'].isna().sum()
print(f'\nBackfilled {mask_2023.sum():,} public 2023 rows from 2021 CCD')
print(f'  hs_magnet still null after backfill:         {still_null_mag:,}')
print(f'  hs_title_i_status still null after backfill: {still_null_ti:,}')

2021 CCD pulled: 102,130 schools
  magnet non-null:         100,425
  title_i_status non-null: 100,425

Backfilled 28,072 public 2023 rows from 2021 CCD
  hs_magnet still null after backfill:         665
  hs_title_i_status still null after backfill: 665


## 2. Pull Race Data for Public HS (CCD Enrollment API)

In [5]:
# Pull total enrollment by race (grade-99 = all grades, sex=99 = all sexes)
# Race codes: 1=White, 2=Black, 3=Hispanic, 4=Asian, 5=AIAN, 6=NHPI, 7=Two or more, 99=Total
race_codes = [1, 2, 3, 4, 5, 6, 7, 99]
all_enroll = []

for year in YEARS:
    for race in race_codes:
        url = (f'https://educationdata.urban.org/api/v1/schools/ccd/enrollment/'
               f'{year}/grade-99/race/?race={race}&sex=99')
        while url:
            data = fetch_json(url)
            for r in data.get('results', []):
                all_enroll.append({'ncessch': r.get('ncessch'), 'year': year,
                                   'race': race, 'enrollment': r.get('enrollment')})
            url = data.get('next')
            time.sleep(0.03)
        print(f'  {year} race={race} done  (total rows so far: {len(all_enroll):,})')

enroll_df = pd.DataFrame(all_enroll)
enroll_df['ncessch']   = enroll_df['ncessch'].astype('string').str.strip()
enroll_df['enrollment'] = pd.to_numeric(enroll_df['enrollment'], errors='coerce')
print('enroll_df shape:', enroll_df.shape)

  2019 race=1 done  (total rows so far: 99,290)
  2019 race=2 done  (total rows so far: 198,580)
  2019 race=3 done  (total rows so far: 297,870)
  2019 race=4 done  (total rows so far: 397,160)
  2019 race=5 done  (total rows so far: 496,450)
  2019 race=6 done  (total rows so far: 595,740)
  2019 race=7 done  (total rows so far: 695,030)
  2019 race=99 done  (total rows so far: 794,320)
  2023 race=1 done  (total rows so far: 893,998)
  2023 race=2 done  (total rows so far: 993,676)
  2023 race=3 done  (total rows so far: 1,093,354)
  2023 race=4 done  (total rows so far: 1,193,032)
  2023 race=5 done  (total rows so far: 1,292,710)
  2023 race=6 done  (total rows so far: 1,392,388)
  2023 race=7 done  (total rows so far: 1,492,066)
  2023 race=99 done  (total rows so far: 1,591,744)
enroll_df shape: (1591744, 4)


In [6]:
# Pivot and compute race percentages (same logic as pull_hs_race_data.ipynb)
pivot = (
    enroll_df.pivot_table(index=['ncessch', 'year'], columns='race',
                          values='enrollment', aggfunc='sum')
    .reset_index()
)
pivot.columns.name = None

total = pivot.get(99, pd.Series(np.nan, index=pivot.index))
pivot['_total'] = total
pivot.loc[pivot['_total'].isna() | (pivot['_total'] == 0), '_total'] = np.nan

race_map = {
    1: 'hs_pct_white', 2: 'hs_pct_black',    3: 'hs_pct_hispanic',
    4: 'hs_pct_asian', 5: 'hs_pct_aian',     6: 'hs_pct_nhpi',
    7: 'hs_pct_two_or_more',
}
for code, col in race_map.items():
    pivot[col] = pivot.get(code, pd.Series(0, index=pivot.index)) / pivot['_total']

hs_race = pivot[['ncessch', 'year'] + list(race_map.values())].copy()
hs_race.rename(columns={'ncessch': 'hs_id'}, inplace=True)

# Filter to only schools in our public HS universe
pub_ids = set(pub_hs['hs_id'])
hs_race = hs_race[hs_race['hs_id'].isin(pub_ids)]

print('hs_race shape:', hs_race.shape)
hs_race.head(3)

hs_race shape: (55877, 9)


,hs_id,year,hs_pct_white,hs_pct_black,hs_pct_hispanic,hs_pct_asian,hs_pct_aian,hs_pct_nhpi,hs_pct_two_or_more
2,010000500871,2019,0.457529,0.039254,0.471042,0.005148,0.001931,0.000000,0.025097
3,010000500871,2023,0.350877,0.039181,0.573684,0.002924,0.003509,0.000585,0.029240
14,010000600872,2019,0.549020,0.014706,0.424837,0.003268,0.003268,0.001634,0.003268


In [7]:
# Merge race data into public HS frame
pub_hs = pub_hs.merge(hs_race, left_on=['hs_id', 'year'], right_on=['hs_id', 'year'], how='left')
print('pub_hs after race merge:', pub_hs.shape)
print(f'Race fill rate (pct_white): {pub_hs["hs_pct_white"].notna().mean():.1%}')

pub_hs after race merge: (55211, 27)
Race fill rate (pct_white): 92.3%


## 3. Pull Private HS from PSS 2019-20\n\nPSS 2019-20 is used for both 2019 and 2023 (same private school universe).\nFilter: `HIGR2020 == 17` (highest grade is 12th).\n\nP_WHITE/P_BLACK/etc. are already percentage values (0-100), divided by 100 to match public.

In [8]:
priv_level_map = {1: 'elementary', 2: 'secondary', 3: 'combined'}

def load_pss(path, higr_col, lat_col, lon_col, cty_col, year):
    """Load PSS file, filter to grade-12 schools, and standardize to hs_ columns."""
    pss = pd.read_csv(path, low_memory=False, usecols=[
        'PPIN', 'PINST', 'PCITY', 'PSTABB', 'PZIP', 'PCNTNM', cty_col,
        lat_col, lon_col, higr_col, 'LEVEL', 'P305', 'NUMTEACH',
        'P_WHITE', 'P_BLACK', 'P_HISP', 'P_ASIAN', 'P_INDIAN', 'P_PACIFIC', 'P_TR',
    ])

    pss[higr_col] = pd.to_numeric(pss[higr_col], errors='coerce')
    pss = pss[pss[higr_col] == 17].copy()  # 17 = highest grade is 12th

    pss['hs_id']   = pss['PPIN'].astype('string').str.strip()
    pss['hs_name'] = pss['PINST'].astype('string').str.strip()
    pss['hs_city'] = pss['PCITY'].astype('string').str.strip()
    pss['hs_state']   = pss['PSTABB'].astype('string').str.strip()
    pss['hs_zip']     = pd.to_numeric(pss['PZIP'], errors='coerce')
    pss['hs_ctyname'] = pss['PCNTNM'].astype('string').str.strip()
    pss['hs_cty_fips']= pd.to_numeric(pss[cty_col], errors='coerce')
    pss['hs_lat']  = pd.to_numeric(pss[lat_col], errors='coerce')
    pss['hs_long'] = pd.to_numeric(pss[lon_col], errors='coerce')
    pss['hs_highest_grade_offered'] = 12.0
    pss['hs_school_level'] = pd.to_numeric(pss['LEVEL'], errors='coerce').map(
        lambda v: priv_level_map.get(int(v)) if pd.notna(v) else None
    )
    pss['hs_enrollment'] = pd.to_numeric(pss['P305'], errors='coerce')
    numteach = pd.to_numeric(pss['NUMTEACH'], errors='coerce')
    pss['hs_students_per_teacher'] = np.where(
        numteach.notna() & (numteach > 0) & pss['hs_enrollment'].notna(),
        pss['hs_enrollment'] / numteach, np.nan
    )

    # P_* are already percentages (0-100) → divide by 100
    for src, dst in [('P_WHITE','hs_pct_white'),('P_BLACK','hs_pct_black'),
                     ('P_HISP','hs_pct_hispanic'),('P_ASIAN','hs_pct_asian'),
                     ('P_INDIAN','hs_pct_aian'),('P_PACIFIC','hs_pct_nhpi'),
                     ('P_TR','hs_pct_two_or_more')]:
        pss[dst] = pd.to_numeric(pss[src], errors='coerce') / 100

    # Public-only fields: -10 = not applicable
    for col in ['hs_title_i_status','hs_urban_centric_locale','hs_charter',
                'hs_magnet','hs_pct_free_or_reduced_price_lunch']:
        pss[col] = -10

    pss['school_type'] = 'private'
    pss['year'] = year

    hs_cols = ['hs_id','year','hs_name','hs_city','hs_state','hs_zip','hs_ctyname',
               'hs_cty_fips','hs_lat','hs_long','school_type','hs_school_level',
               'hs_highest_grade_offered','hs_charter','hs_magnet','hs_title_i_status',
               'hs_urban_centric_locale','hs_enrollment','hs_students_per_teacher',
               'hs_pct_white','hs_pct_black','hs_pct_hispanic','hs_pct_asian',
               'hs_pct_aian','hs_pct_nhpi','hs_pct_two_or_more',
               'hs_pct_free_or_reduced_price_lunch']

    return pss[hs_cols].drop_duplicates(subset=['hs_id'])

# Load PSS 2019-20 once and stamp for both years
pss_base = load_pss(
    DATA_DIR / 'pss_2019-20.csv',
    higr_col='HIGR2020', lat_col='LATITUDE20', lon_col='LONGITUDE20', cty_col='PCNTY20',
    year=2019  # year will be overridden below
)

priv_2019 = pss_base.copy()
priv_2019['year'] = 2019

priv_2023 = pss_base.copy()
priv_2023['year'] = 2023

print(f'Private HS (PSS 2019-20): {len(pss_base):,} unique schools')
print(f'  Used for 2019: {len(priv_2019):,} rows')
print(f'  Used for 2023: {len(priv_2023):,} rows')

Private HS (PSS 2019-20): 6,731 unique schools
  Used for 2019: 6,731 rows
  Used for 2023: 6,731 rows


## 4. Combine Public + Private into Full Universe

In [9]:
# Align public HS columns to the same set
universe_cols = [
    'hs_id', 'year', 'hs_name', 'hs_city', 'hs_state', 'hs_zip', 'hs_ctyname',
    'hs_cty_fips', 'hs_lat', 'hs_long', 'school_type', 'hs_school_level',
    'hs_highest_grade_offered', 'hs_charter', 'hs_magnet', 'hs_title_i_status',
    'hs_urban_centric_locale', 'hs_enrollment', 'hs_students_per_teacher',
    'hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic', 'hs_pct_asian',
    'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more',
    'hs_pct_free_or_reduced_price_lunch',
]

pub_aligned = pub_hs.reindex(columns=universe_cols)
priv_aligned = pd.concat([priv_2019, priv_2023], ignore_index=True).reindex(columns=universe_cols)

universe = pd.concat([pub_aligned, priv_aligned], ignore_index=True)
universe['hs_id'] = universe['hs_id'].astype('string').str.strip()
universe = universe.drop_duplicates(subset=['hs_id', 'year'])

for yr in YEARS:
    sub = universe[universe['year'] == yr]
    print(f'{yr}: {len(sub):,} total HS  '
          f'({(sub["school_type"]=="public").sum():,} public, '
          f'{(sub["school_type"]=="private").sum():,} private)')

2019: 33,870 total HS  (27,139 public, 6,731 private)
2023: 34,803 total HS  (28,072 public, 6,731 private)


## 5. Merge with hs_in_degree — Set Unvisited Schools to 0

In [10]:
hs_indegree = pd.read_csv(DATA_DIR / 'hs_in_degree.csv')
hs_indegree['hs_id'] = hs_indegree['hs_id'].astype('string').str.strip()
hs_indegree['year']  = pd.to_numeric(hs_indegree['year'], errors='coerce')

print(f'hs_in_degree rows: {len(hs_indegree):,}')
for yr in YEARS:
    print(f'  {yr}: {(hs_indegree["year"]==yr).sum():,} schools')

hs_in_degree rows: 40,187
  2019: 21,692 schools
  2023: 18,495 schools


In [11]:
# Merge: left join from universe onto in-degree (universe is the full set)
# Schools in hs_in_degree get their n_colleges_visiting value;
# schools not in hs_in_degree get n_colleges_visiting = 0

degree_slim = hs_indegree[['hs_id', 'year', 'n_colleges_visiting']].copy()

merged = universe.merge(degree_slim, on=['hs_id', 'year'], how='left')
merged['n_colleges_visiting'] = merged['n_colleges_visiting'].fillna(0).astype(int)

# Put n_colleges_visiting right after year
cols = merged.columns.tolist()
cols.remove('n_colleges_visiting')
cols.insert(cols.index('year') + 1, 'n_colleges_visiting')
merged = merged[cols]

print(f'Final dataset: {len(merged):,} rows')
for yr in YEARS:
    sub = merged[merged['year'] == yr]
    visited = (sub['n_colleges_visiting'] > 0).sum()
    print(f'  {yr}: {len(sub):,} HS total | {visited:,} visited ({visited/len(sub):.1%}) '
          f'| {len(sub)-visited:,} unvisited')

Final dataset: 68,673 rows
  2019: 33,870 HS total | 21,692 visited (64.0%) | 12,178 unvisited
  2023: 34,803 HS total | 18,495 visited (53.1%) | 16,308 unvisited


## 6. Save

In [12]:
out_path = DATA_DIR / 'hs_universe_indegree.csv'
merged.to_csv(out_path, index=False)
print(f'Saved {out_path}  ({len(merged):,} rows x {len(merged.columns)} cols)')
print('Columns:', merged.columns.tolist())

Saved /Users/bryceclement/Desktop/c2i/data/hs_universe_indegree.csv  (68,673 rows x 28 cols)
Columns: ['hs_id', 'year', 'n_colleges_visiting', 'hs_name', 'hs_city', 'hs_state', 'hs_zip', 'hs_ctyname', 'hs_cty_fips', 'hs_lat', 'hs_long', 'school_type', 'hs_school_level', 'hs_highest_grade_offered', 'hs_charter', 'hs_magnet', 'hs_title_i_status', 'hs_urban_centric_locale', 'hs_enrollment', 'hs_students_per_teacher', 'hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic', 'hs_pct_asian', 'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more', 'hs_pct_free_or_reduced_price_lunch']
